# Sense-Think-Act Assignment

The goal of this assignment is to implement a simple robot control algorithm that can detect light intensity and spin the robot in response to high intensity light.



## Method

To complete the assignment, you must first break down the project into smaller, manageable components.​

1. Reading the ambient light sensor​
1. Identifying the threshold LUX value for the robot spin​
1. Controlling the motors (actuators) on RVR​
1. Spinning the RVR​
1. Logic to connect sensor input to the actuation output​
1. Putting it all together within a ‘Sense, Think, Act’ loop

#### Light Sensor

The light sensor is located on the front of the RVR. It is a digital sensor that returns a number. The higher the value, the brighter the light.​

#### Threshold Value

The threshold value is the value at which the robot will spin. You will need to experiment with different values to find the best one.​

#### Motor Control

The rover responds to twist commands. The twist command is a combination of linear and angular velocity. The linear velocity is the speed at which the robot moves forward or backward. The angular velocity is the speed at which the robot spins.​

The `cmd_vel` topic is a standard topic used to send twist commands to a robot. The message type is `geometry_msgs/Twist`.​

In [1]:

"""
setting up ros enrionment variables
ROS 2 networking setup for shared Wi‑Fi with discovery server
"""

import os

#####################################################################
# # !!!! DON'T FORGET TO CHANGE THIS TO THE ROBOT'S IP ADDRESS !!!! #
#####################################################################

# change the ROS environment variable to the robot's IP address
ROBOT_HOSTNAME = "192.168.1.76"     # robot (or discovery server) IP/hostname
# for example, this could be an IP or a hostname depending on what is reachable on the network
CONSOLE_HOSTNAME = "192.168.1.208"   # this laptop/container IP/hostname
# for example, notice that this is the IP of the computer running this script not the robot

# another example using hostnames is as follows
# ROBOT_HOSTNAME = 'rvr-001' # for example, this could be an IP or a hostname depending on what is reachable on the network
# CONSOLE_HOSTNAME = 'ireslab-group1' # for example, notice that this is the hostname of the computer running this script not the robot

DISCOVERY_SERVER_PORT = 11811        # fastdds discovery server port
DISCOVERY_SERVER_HOST = ROBOT_HOSTNAME
ROS_DOMAIN_ID = "8"                 # the ROS network unique domain ID per robot - change this to match the robot's domain ID

# the following lines set up the networking variables for the scripts to run properly
# ROS 2 networking (Fast DDS discovery server)
os.environ["ROS_DOMAIN_ID"] = ROS_DOMAIN_ID           # isolates traffic by domain
os.environ["ROS_HOSTNAME"] = CONSOLE_HOSTNAME         # advertises the correct address
os.environ["RMW_IMPLEMENTATION"] = "rmw_fastrtps_cpp" # ensure Fast DDS middleware is used
os.environ["ROS_AUTOMATIC_DISCOVERY_RANGE"] = "SUBNET" # only discover nodes on the same subnet
os.environ["ROS_DISCOVERY_SERVER"] = f"{DISCOVERY_SERVER_HOST}:{DISCOVERY_SERVER_PORT}"  # discovery server location
os.environ["ROS_SUPER_CLIENT"] = "True"                # enable super client mode

In [2]:
# ROS 2 uses a communication middleware called RMW (ROS Middleware)
# to start RMW correctly, we need to initialize rclpy
import rclpy  # Initialize rclpy once at the top

# Initialize rclpy once - this must be done before creating any nodes
if not rclpy.ok():
    rclpy.init()
    print("rclpy initialised ✓")
else:
    print("rclpy already running ✓")

rclpy initialised ✓


### 1. Testing Reading the ambient light sensor

In [3]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Illuminance


class LightSensorReaderNode(Node):
    """Subscribes to the illuminance topic and prints each reading."""

    def __init__(self):
        super().__init__('light_sensor_reader_node')
        self.light_sub = self.create_subscription(
            Illuminance,
            '/ambient_light_broadcaster/ambient_light',
            self.light_callback,
            10
        )
        self.get_logger().info('LightSensorReaderNode started — listening for illuminance...')

    def light_callback(self, msg: Illuminance):
        """Called every time a new illuminance message arrives."""
        lux = msg.illuminance
        print(f'Illuminance: {lux:.2f} lux')


def main():
    print('Starting LightSensorReaderNode...')
    try:
        node = LightSensorReaderNode()
        rclpy.spin(node)          # blocks; prints readings as they arrive
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        print('Node shutdown.')


print('LightSensorReaderNode declared — call main() to run it')


LightSensorReaderNode declared — call main() to run it


In [4]:
main()

Starting LightSensorReaderNode...


[INFO] [1773189904.961797762] [light_sensor_reader_node]: LightSensorReaderNode started — listening for illuminance...


Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 407.47 lux
Illuminance: 407.47 lux
Illuminance: 407.47 lux
Illuminance: 407.47 lux
Illuminance: 407.47 lux
Illuminance: 406.49 lux
Illuminance: 406.49 lux
Illuminance: 406.49 lux
Illuminance: 406.49 lux
Illuminance: 406.49 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 419.23 lux
Illuminance: 419.23 lux
Illuminance: 419.23 lux
Illuminance: 419.23 lux
Illuminance: 419.23 lux
Illuminance: 416.29 lux
Illuminance: 416.29 lux
Illuminance: 416.29 lux
Illuminance: 416.29 lux
Illuminance: 416.29 lux
Illuminance: 405.51 lux
Illuminance: 405.51 lux
Illuminance: 405.51 lux
Illuminance: 405.51 lux
Illuminance: 405.51 lux
Illuminance: 408.45 lux
Illuminance: 408.45 lux
Illuminance: 408.45 lux
Illuminance: 408.45 lux
Illuminance: 408.45 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 418.25 lux
Illuminance: 420

### Thershold LUX Value

After testing, the best LUX value  that was determined was 775

### TESTING MOTORS

In [3]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import TwistStamped
from time import sleep


class SpinTestNode(Node):
    """Publishes a fixed spin command at 2 Hz to test motor control."""

    ANGULAR_SPEED = 10.0   # rad/s — change sign to reverse spin direction
    PUBLISH_RATE  = 2.0   # Hz

    def __init__(self):
        super().__init__('spin_test_node')
        self.cmd_vel_pub = self.create_publisher(TwistStamped, '/cmd_vel', 10)
        self.get_logger().info('SpinTestNode started')

    def run(self):
        """Publish spin command in a manual loop."""
        print(f'Spinning at {self.ANGULAR_SPEED} rad/s — press Stop / restart kernel to halt')
        while rclpy.ok():
            msg = TwistStamped()
            msg.twist.linear.x  = 5.0
            msg.twist.angular.z = self.ANGULAR_SPEED
            self.cmd_vel_pub.publish(msg)
            sleep(1.0 / self.PUBLISH_RATE)

    def stop(self):
        """Publish a zero-velocity command to halt the robot."""
        stop_msg = TwistStamped()
        stop_msg.twist.linear.x  = 0.0
        stop_msg.twist.angular.z = 0.0
        self.cmd_vel_pub.publish(stop_msg)
        print('Stop command sent ✓')


def main():
    print('Starting SpinTestNode...')
    node = SpinTestNode()
    try:
        node.run()
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        node.stop()
        print('Node shutdown.')


print('SpinTestNode declared — call main() to run it')


SpinTestNode declared — call main() to run it


In [4]:
main()

Starting SpinTestNode...
Spinning at 1.0 rad/s — press Stop / restart kernel to halt


[INFO] [1773191939.141090563] [spin_test_node]: SpinTestNode started


Interrupted
Stop command sent ✓
Node shutdown.


### WORKING DEMO

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Illuminance
from geometry_msgs.msg import TwistStamped
from rclpy.time import Time


class SenseThinkActNode(Node):

    LIGHT_THRESHOLD = 750.0   # lux — spin when illuminance exceeds this
    ANGULAR_SPEED   = 1.5     # rad/s — spin speed (positive = CCW)
    LOOP_RATE_HZ    = 10.0    # Hz — how often the think/act timer fires

    def __init__(self):
        super().__init__('sense_think_act_node')

        # ---- SENSE: light sensor subscriber ---- #
        self.light_sub = self.create_subscription(
            Illuminance,
            '/ambient_light_broadcaster/ambient_light',
            self._sense_cb,       # called every time a new reading arrives
            10
        )

        # ---- ACT: velocity command publisher ---- #
        self.cmd_vel_pub = self.create_publisher(TwistStamped, '/cmd_vel', 10)

        # ---- THINK: timer drives the control loop ---- #
        self.timer = self.create_timer(
            1.0 / self.LOOP_RATE_HZ,
            self._think_and_act_cb
        )

        # State: latest illuminance reading (starts at 0 — robot is still)
        self.current_lux: float = 0.0

        self.get_logger().info(
            f'SenseThinkActNode started | threshold={self.LIGHT_THRESHOLD} lux | '
            f'spin speed={self.ANGULAR_SPEED} rad/s | loop={self.LOOP_RATE_HZ} Hz'
        )


        # ROS timer ( inseconds) when the spin should stop
        self.spin_end_time: float = 0.0  # 

    def _sense_cb(self, msg: Illuminance):

        self.current_lux = msg.illuminance
        self.get_logger().debug(f'Sensed: {self.current_lux:.1f} lux')

    def _think_and_act_cb(self):
        cmd = TwistStamped()

        # Get the current ROS time in seconds
        now = self.get_clock().now().nanoseconds / 1e9

        # ---- THINK ---- #
        light_is_bright = self.current_lux > self.LIGHT_THRESHOLD

        if light_is_bright:
            # Light detected — (re)start the 5-second spin window
            self.spin_end_time = now + 5.0
            self.get_logger().info(
                f'BRIGHT ({self.current_lux:.1f} lux > {self.LIGHT_THRESHOLD}) '
                f'→ spin window reset, stopping at t={self.spin_end_time:.1f}'
            )

        should_spin = now < self.spin_end_time

        if should_spin:
            # ---- ACT: spin ---- #
            cmd.twist.linear.x  = 0.0
            cmd.twist.angular.z = self.ANGULAR_SPEED
            self.get_logger().info(
                f'SPINNING — {self.spin_end_time - now:.1f}s remaining'
            )
        else:
            # ---- ACT: stop ---- #
            cmd.twist.linear.x  = 0.0
            cmd.twist.angular.z = 0.0
            self.get_logger().info(
                f'DIM ({self.current_lux:.1f} lux) → STOPPED'
            )

        self.cmd_vel_pub.publish(cmd)

    # ============================================================== #
    # CLEANUP                                                         #
    # ============================================================== #
    def stop_robot(self):
        """Publish a zero-velocity command to safely halt the robot."""
        stop_cmd = TwistStamped()
        stop_cmd.twist.linear.x  = 0.0
        stop_cmd.twist.angular.z = 0.0
        self.cmd_vel_pub.publish(stop_cmd)
        self.get_logger().info('Safety stop sent ✓')


# ------------------------------------------------------------------ #
# Main                                                                #
# ------------------------------------------------------------------ #
def main():
    print('Starting SenseThinkActNode...')
    print(f'  Threshold : {SenseThinkActNode.LIGHT_THRESHOLD} lux')
    print(f'  Spin speed: {SenseThinkActNode.ANGULAR_SPEED} rad/s')
    print(f'  Loop rate : {SenseThinkActNode.LOOP_RATE_HZ} Hz')
    print()

    node = SenseThinkActNode()
    try:
        # rclpy.spin() drives both the subscriber callback and the timer callback
        rclpy.spin(node)
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        node.stop_robot()   # always stop the robot on exit
        print('Node shutdown.')


print('SenseThinkActNode declared ✓')



SenseThinkActNode declared ✓


In [ ]:
main()

Starting SenseThinkActNode...
  Threshold : 750.0 lux
  Spin speed: 1.5 rad/s
  Loop rate : 10.0 Hz



[INFO] [1773193361.638520045] [sense_think_act_node]: SenseThinkActNode started | threshold=750.0 lux | spin speed=1.5 rad/s | loop=10.0 Hz
[INFO] [1773193361.682894383] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193361.783439794] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193361.883292997] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193361.982975325] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.083106941] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.184211302] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.282615455] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.382378110] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.484267153] [sense_think_act_node]: DIM   (0.0 lux ≤ 750.0) → STOPPED
[INFO] [1773193362.583261901] [sense_think_act_node]: DIM   (146.7 l